# Notebook 3: Supplementary data

Enriching the 50,718 HDB resale transactions with location distances (from OneMap), numerology variables, and cultural timing. All geocoding hit 100% coverage across 9,118 unique blocks.

In [1]:
import pandas as pd
import numpy as np
import os, requests, time, re

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

def haversine_km(lat1, lon1, lat2, lon2):
    """Haversine distance in kilometres."""
    R = 6371
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

print('Setup complete.')

Setup complete.


## Geocode HDB blocks

Query each unique block + street combo against OneMap and cache the lat/lon results.

In [2]:
CACHE_FILE = 'data/onemap_geocoded.csv'

if os.path.exists(CACHE_FILE):
    geo = pd.read_csv(CACHE_FILE)
    print(f'Loaded cached geocoding: {len(geo):,} block+street combos')
else:
    hdb_raw = pd.read_csv('data/hdb_resale_cleaned.csv', usecols=['block', 'street_name'])
    combos = hdb_raw.drop_duplicates().reset_index(drop=True)
    print(f'Geocoding {len(combos):,} unique block+street combos via OneMap API...')
    
    results = []
    errors = 0
    for i, row in combos.iterrows():
        query = f"{row['block']} {row['street_name']}"
        try:
            resp = requests.get(
                'https://www.onemap.gov.sg/api/common/elastic/search',
                params={'searchVal': query, 'returnGeom': 'Y', 'getAddrDetails': 'Y'},
                timeout=10
            )
            data = resp.json()
            if data.get('found', 0) > 0:
                r = data['results'][0]
                results.append({'block': row['block'], 'street_name': row['street_name'],
                    'latitude': float(r['LATITUDE']), 'longitude': float(r['LONGITUDE'])})
            else:
                results.append({'block': row['block'], 'street_name': row['street_name'],
                    'latitude': None, 'longitude': None})
                errors += 1
        except Exception:
            results.append({'block': row['block'], 'street_name': row['street_name'],
                'latitude': None, 'longitude': None})
            errors += 1
        if (i + 1) % 500 == 0:
            print(f'  {i+1:,} / {len(combos):,} done ({errors} errors)')
        if (i + 1) % 50 == 0:
            time.sleep(0.5)
    
    geo = pd.DataFrame(results)
    geo.to_csv(CACHE_FILE, index=False)
    print(f'Done! {geo["latitude"].notna().sum():,} / {len(combos):,} geocoded')

matched = geo['latitude'].notna().sum()
print(f'\nGeocoding coverage: {matched:,} / {len(geo):,} ({matched/len(geo)*100:.1f}%)')

Loaded cached geocoding: 9,532 block+street combos

Geocoding coverage: 9,532 / 9,532 (100.0%)


## Distance to nearest locations

Haversine distance from each block to the nearest point in each category:

| Category | File | Count | Feng shui valence |
|---|---|---|---|
| MRT/LRT stations | `mrt_stations.csv` | 783 | Neutral (amenity) |
| Schools | `schools.csv` | 310 | Neutral (amenity) |
| Hawker centres | `loc_hawker_centres.csv` | 128 | Neutral (amenity) |
| Supermarkets | `loc_supermarkets.csv` | 22 | Neutral (amenity) |
| Parks | `loc_parks.csv` | 87 | Positive (nature) |
| Reservoirs | `loc_reservoirs.csv` | 50 | Positive (water = wealth) |
| Coastline | `loc_coast.csv` | 19 | Positive (waterfront) |
| Hospitals | `loc_hospitals.csv` | 70 | Negative (yin) vs positive (amenity) |
| Temples | `loc_temples.csv` | 292 | Mixed |
| Columbaria/cemeteries | `loc_columbarium.csv` | 12 | Very negative |
| Funeral parlours | `loc_funeral.csv` | 3 | Very negative |
| CBD (Raffles Place) | Fixed point | 1 | N/A |

In [3]:
# Load HDB cleaned data and join geocoding
hdb = pd.read_csv('data/hdb_resale_cleaned.csv')
hdb = hdb.merge(geo[['block', 'street_name', 'latitude', 'longitude']],
                on=['block', 'street_name'], how='left')
print(f'HDB transactions: {len(hdb):,}')
print(f'Geocoded: {hdb["latitude"].notna().sum():,} ({hdb["latitude"].notna().mean()*100:.1f}%)')

# Load all location reference files
loc_files = {
    'mrt': ('data/mrt_stations.csv', 'MRT/LRT stations'),
    'school': ('data/schools.csv', 'Schools'),
    'hawker': ('data/loc_hawker_centres.csv', 'Hawker centres'),
    'supermarket': ('data/loc_supermarkets.csv', 'Supermarkets'),
    'park': ('data/loc_parks.csv', 'Parks'),
    'columbarium': ('data/loc_columbarium.csv', 'Columbaria/cemeteries'),
    'funeral': ('data/loc_funeral.csv', 'Funeral parlours'),
    'hospital': ('data/loc_hospitals.csv', 'Hospitals'),
    'temple': ('data/loc_temples.csv', 'Temples/worship'),
    'reservoir': ('data/loc_reservoirs.csv', 'Reservoirs'),
    'coast': ('data/loc_coast.csv', 'Coastline'),
    'popular_school': ('data/loc_popular_schools.csv', 'Oversubscribed primary schools'),
}

locations = {}
for key, (path, label) in loc_files.items():
    df = pd.read_csv(path)
    locations[key] = df
    print(f'  {label}: {len(df):,}')

# --- Distance to CBD ---
CBD_LAT, CBD_LON = 1.2840, 103.8514
hdb['dist_cbd_km'] = haversine_km(hdb['latitude'], hdb['longitude'], CBD_LAT, CBD_LON)

# --- Compute nearest distance for each location type ---
# Do this per unique block to avoid redundant computation
block_coords = hdb[['block', 'street_name', 'latitude', 'longitude']].drop_duplicates()
block_coords = block_coords.dropna(subset=['latitude'])
print(f'\nComputing distances for {len(block_coords):,} unique blocks to {len(locations)} location types...')

for key, ref_df in locations.items():
    dist_col = f'{key}_dist_m'
    name_col = f'nearest_{key}_name'
    
    ref_lats = ref_df['latitude'].values
    ref_lons = ref_df['longitude'].values
    ref_names = ref_df['name'].values
    
    dists = []
    names = []
    for _, row in block_coords.iterrows():
        d = haversine_km(row['latitude'], row['longitude'], ref_lats, ref_lons) * 1000
        idx = np.argmin(d)
        dists.append(d[idx])
        names.append(ref_names[idx])
    
    block_coords[dist_col] = dists
    block_coords[name_col] = names
    print(f'  {key}: median {np.median(dists):.0f}m')

# Join back to HDB data
join_cols = ['block', 'street_name'] + [c for c in block_coords.columns 
             if c.endswith('_dist_m') or c.startswith('nearest_') and c != 'nearest_mrt_name']
# Filter to only new distance/name columns
new_cols = [c for c in block_coords.columns if '_dist_m' in c or 'nearest_' in c]
hdb = hdb.merge(block_coords[['block', 'street_name'] + new_cols],
                on=['block', 'street_name'], how='left')

print(f'\nAll distances computed and joined.')


HDB transactions: 50,718
Geocoded: 50,718 (100.0%)
  MRT/LRT stations: 783
  Schools: 310
  Hawker centres: 128
  Supermarkets: 22
  Parks: 87
  Columbaria/cemeteries: 12
  Funeral parlours: 3
  Hospitals: 70
  Temples/worship: 292
  Reservoirs: 50
  Coastline: 19
  Oversubscribed primary schools: 23

Computing distances for 9,118 unique blocks to 12 location types...


  mrt: median 452m
  school: median 307m


  hawker: median 696m
  supermarket: median 1887m


  park: median 1527m
  columbarium: median 4033m


  funeral: median 4270m
  hospital: median 1426m


  temple: median 574m
  reservoir: median 2442m


  coast: median 3427m
  popular_school: median 1966m

All distances computed and joined.


## Superstition variables

Testing how superstitious Singapore's property pricing really is.

- `num_eights_in_price` / `num_fours_in_price` -- count of 8s and 4s in the price
- `price_has_888` -- triple-8 anywhere in price
- `price_has_444` -- triple-4 (maximally unlucky)
- `price_has_168` -- 一路发 "prosperity all the way"
- `price_has_138` -- 一生发 "prosperity for life"
- `price_has_328` -- 生意发 "business prospers"
- `has_floor_4` -- storey range includes floor 4 (四 sounds like 死 "death")
- `has_floor_13` -- Western unlucky number
- `has_floor_14` -- in some dialects sounds like 实死 "sure die"
- `block_has_4`, `block_has_8`, `block_num_eights` -- digit composition of block number

In [4]:
# === Price digit variables ===
price_str = hdb['resale_price'].astype(int).astype(str)

hdb['num_eights_in_price'] = price_str.str.count('8')
hdb['num_fours_in_price'] = price_str.str.count('4')
hdb['price_has_888'] = price_str.str.contains('888').astype(int)
hdb['price_has_444'] = price_str.str.contains('444').astype(int)
hdb['price_has_168'] = price_str.str.contains('168').astype(int)
hdb['price_has_138'] = price_str.str.contains('138').astype(int)
hdb['price_has_328'] = price_str.str.contains('328').astype(int)

# Tail-only versions: count 8s and 4s in LAST 4 digits only
# Avoids counting leading digits that reflect price level, not seller choice
price_tail = price_str.str[-4:]
hdb['num_eights_tail'] = price_tail.str.count('8')
hdb['num_fours_tail'] = price_tail.str.count('4')

# === Floor superstition variables ===
storey_low = hdb['storey_range'].str.extract(r'(\d+)\s+TO').astype(float)[0]
storey_high = hdb['storey_range'].str.extract(r'TO\s+(\d+)').astype(float)[0]
hdb['has_floor_4'] = ((storey_low <= 4) & (storey_high >= 4)).astype(int)
hdb['has_floor_13'] = ((storey_low <= 13) & (storey_high >= 13)).astype(int)
hdb['has_floor_14'] = ((storey_low <= 14) & (storey_high >= 14)).astype(int)

# === Block number variables ===
hdb['block_has_4'] = hdb['block'].str.contains('4').astype(int)
hdb['block_has_8'] = hdb['block'].str.contains('8').astype(int)
hdb['block_num_eights'] = hdb['block'].str.count('8')

print('=== Price digit distributions ===\n')
print(f'num_eights_in_price:')
print(hdb['num_eights_in_price'].value_counts().sort_index())
print(f'\nnum_fours_in_price:')
print(hdb['num_fours_in_price'].value_counts().sort_index())
print(f'\nAuspicious patterns:')
for v in ['price_has_888', 'price_has_168', 'price_has_138', 'price_has_328']:
    print(f'  {v}: {hdb[v].sum():,} ({hdb[v].mean()*100:.2f}%)')
print(f'\nInauspicious:')
print(f'  price_has_444: {hdb["price_has_444"].sum():,} ({hdb["price_has_444"].mean()*100:.2f}%)')
print(f'\n=== Floor superstition ===')
for v in ['has_floor_4', 'has_floor_13', 'has_floor_14']:
    print(f'  {v}: {hdb[v].sum():,} ({hdb[v].mean()*100:.1f}%)')
print(f'\n=== Block numerology ===')
for v in ['block_has_4', 'block_has_8']:
    print(f'  {v}: {hdb[v].sum():,} ({hdb[v].mean()*100:.1f}%)')

=== Price digit distributions ===

num_eights_in_price:
num_eights_in_price
0    31699
1    12415
2     2041
3     2480
4     1686
5      367
6       30
Name: count, dtype: int64

num_fours_in_price:
num_fours_in_price
0    38981
1    11091
2      643
3        2
4        1
Name: count, dtype: int64

Auspicious patterns:
  price_has_888: 4,485 (8.84%)
  price_has_168: 103 (0.20%)
  price_has_138: 82 (0.16%)
  price_has_328: 100 (0.20%)

Inauspicious:
  price_has_444: 2 (0.00%)

=== Floor superstition ===
  has_floor_4: 11,506 (22.7%)
  has_floor_13: 5,060 (10.0%)
  has_floor_14: 5,060 (10.0%)

=== Block numerology ===
  block_has_4: 14,168 (27.9%)
  block_has_8: 10,551 (20.8%)


## Transaction timing

- **Hungry Ghost Month** (7th lunar month) -- inauspicious for major purchases
- **CNY month** (1st lunar month) -- festive period, fewer transactions

| Year | Hungry Ghost Month | CNY Month |
|------|-------------------|----------|
| 2024 | 4 Aug -- 2 Sep | 10 Feb -- 9 Mar |
| 2025 | 22 Aug -- 20 Sep | 29 Jan -- 27 Feb |
| 2026 | 12 Sep -- 10 Oct | 17 Feb -- 18 Mar |

In [5]:
ghost_ranges = [
    ('2024-08-04', '2024-09-02'),
    ('2025-08-22', '2025-09-20'),
    ('2026-09-12', '2026-10-10'),
]
cny_ranges = [
    ('2024-02-10', '2024-03-09'),
    ('2025-01-29', '2025-02-27'),
    ('2026-02-17', '2026-03-18'),
]

hdb['month_dt'] = pd.to_datetime(hdb['month'])

hdb['hungry_ghost'] = 0
for start, end in ghost_ranges:
    mask = (hdb['month_dt'] >= start) & (hdb['month_dt'] <= end)
    hdb.loc[mask, 'hungry_ghost'] = 1

hdb['cny_month'] = 0
for start, end in cny_ranges:
    mask = (hdb['month_dt'] >= start) & (hdb['month_dt'] <= end)
    hdb.loc[mask, 'cny_month'] = 1

hdb = hdb.drop(columns=['month_dt'])

print(f'Hungry Ghost Month: {hdb["hungry_ghost"].sum():,} ({hdb["hungry_ghost"].mean()*100:.1f}%)')
print(f'CNY Month:          {hdb["cny_month"].sum():,} ({hdb["cny_month"].mean()*100:.1f}%)')
print(f'\n(Note: HDB month is registration month — may lag actual transaction by 1-2 months.)')

Hungry Ghost Month: 4,378 (8.6%)
CNY Month:          4,142 (8.2%)

(Note: HDB month is registration month — may lag actual transaction by 1-2 months.)


## Coverage summary and save

In [6]:
# All supplementary columns
supp_cols = [c for c in hdb.columns if c not in 
    ['month', 'town', 'block', 'street_name', 'resale_price', 'log_resale_price',
     'flat_type', 'flat_model', 'floor_area_sqm', 'storey_range', 'storey_mid',
     'lease_commence_date', 'remaining_lease_years', 'remaining_lease',
     'ends_in_8', 'last_digit', 'dataset']]

print(f'=== Coverage of {len(supp_cols)} supplementary variables ===\n')
for col in sorted(supp_cols):
    n = hdb[col].notna().sum()
    pct = n / len(hdb) * 100
    print(f'  {col:30s}: {n:>6,} / {len(hdb):,} ({pct:.1f}%)')

print(f'\n=== Final dataset ===')
print(f'Rows: {len(hdb):,}')
print(f'Columns: {len(hdb.columns)}')

hdb.to_csv('data/hdb_resale_supplementary.csv', index=False)
print(f'\nSaved to data/hdb_resale_supplementary.csv')


=== Coverage of 44 supplementary variables ===

  block_has_4                   : 50,718 / 50,718 (100.0%)
  block_has_8                   : 50,718 / 50,718 (100.0%)
  block_num_eights              : 50,718 / 50,718 (100.0%)
  cny_month                     : 50,718 / 50,718 (100.0%)
  coast_dist_m                  : 50,718 / 50,718 (100.0%)
  columbarium_dist_m            : 50,718 / 50,718 (100.0%)
  dist_cbd_km                   : 50,718 / 50,718 (100.0%)
  funeral_dist_m                : 50,718 / 50,718 (100.0%)
  has_floor_13                  : 50,718 / 50,718 (100.0%)
  has_floor_14                  : 50,718 / 50,718 (100.0%)
  has_floor_4                   : 50,718 / 50,718 (100.0%)
  hawker_dist_m                 : 50,718 / 50,718 (100.0%)
  hospital_dist_m               : 50,718 / 50,718 (100.0%)
  hungry_ghost                  : 50,718 / 50,718 (100.0%)
  latitude                      : 50,718 / 50,718 (100.0%)
  longitude                     : 50,718 / 50,718 (100.0%)
  mrt_di


Saved to data/hdb_resale_supplementary.csv


## Limitations

- All distances are straight-line haversine, not walking distance. A 500m crow-flies distance could be 800m on foot.
- Station list includes future MRT stations (e.g. Cross Island Line) that aren't open yet -- could inflate apparent MRT proximity for some blocks.
- School list (310) is mostly primary and secondary. Preschools, international schools, tertiary institutions may be missing.
- HDB `month` is registration month, which can lag the actual transaction by 1-2 months. Makes Hungry Ghost Month a rough proxy.
- Block numbers are assigned by HDB, not chosen by buyers. But `has_floor_4` is endogenous -- superstitious buyers may avoid floor 4, so the effect captures both superstition and selection.